In [ ]:
#first import libraries
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont, ImageEnhance
import os

#tensorflow
import tensorflow as tf
#from tensorflow.python.keras.utils.vis_utils import plot_model

#import time
import time

We will use images of Waldo, Wilma and a background image in order to synthasize example images and feed into our deep neural network. The images are created by placing either Waldo or Wilma in a random location of the background image. 

FIrst let's see the individual images of Waldo, Wilma and the background image and then let's create an example image to see what it looks like. 

In [ ]:
#get the images directory
image_dir = '/kaggle/input/waldowilma/generated_images'

#get the background (any image from orginal_350x500) and waldo image directory
background_dir = '/kaggle/input/waldowilma/generated_images/original_350x500/image_0.png'
waldo_dir = image_dir + '/charecters/waldo.png'
wilma_dir = image_dir + '/charecters/wilma.png'

#image sizes. We work at half the original 500x350 resolution: 4x fewer pixels means
#roughly 4x less work in every convolution, and Waldo is still easy to see at 30x50
IMG_W, IMG_H = 250, 175
CHAR_W, CHAR_H = 30, 50

In [ ]:
#background image
background_im = Image.open(background_dir)
background_im

In [ ]:
#image of waldo
waldo_im = Image.open(waldo_dir)
waldo_im = waldo_im.resize((CHAR_W, CHAR_H))
waldo_im

In [ ]:
#wilma
wilma_im = Image.open(wilma_dir)
wilma_im = wilma_im.resize((CHAR_W, CHAR_H))
wilma_im

<u> **Generating sample images** </u>

In [ ]:
#load and resize the images ONCE. Opening PNG files from disk for every single
#sample was one of the slowest parts of training
background_base = Image.open(background_dir).convert('RGB').resize((IMG_W, IMG_H))
waldo_base = Image.open(waldo_dir).convert('RGBA').resize((CHAR_W, CHAR_H))
wilma_base = Image.open(wilma_dir).convert('RGBA').resize((CHAR_W, CHAR_H))

#create a function to generate images
def generate_sample_image():
    
    #copy the background so we don't paste onto the original
    background_im = background_base.copy()
    
    #select x and y coordinates randomly so the person always fits inside the image
    col = np.random.randint(0, IMG_W - CHAR_W)
    row = np.random.randint(0, IMG_H - CHAR_H)
    
    #pic randomly between waldo and wilma. If 1 we will select waldo. if 0 we wills elect wilma
    rand_person = np.random.choice([0, 1], p = [0.5, 0.5])
    
    if rand_person == 1:
        
        background_im.paste(waldo_base, (col, row), mask = waldo_base)
        cat = 'Waldo'
        
    else:
        
        background_im.paste(wilma_base, (col, row), mask = wilma_base)
        cat = 'Wilma'
        
    return np.array(background_im).astype('uint8'), (col, row), rand_person, cat 

In [ ]:
#generate the sample image and plot
sample_im, pos, _, cat = generate_sample_image()
plt.imshow(sample_im)
plt.xticks([])
plt.yticks([])
plt.show()

We see an example image above. In the image we will see either Waldo or Wilma is placed in a random location and we will try to use deep learning to find the location and the person (if its Wilma or Waldo).

See for yourself if you can find Waldo or Wilma in the example above.

<u> **Creating Bounding Boxes** </u>


Next we will create a function to plot a bounding box around the person. We will use a green box to show the real box about the person and we will use a red box to show the predicted location by the model. (Note that the red box in the picture below is created for illustration purposes. Later we will use the model predicted location to place the red box)

In [ ]:
def plot_bounding_box(image, gt_coords, pred_coords = None):
    
    #convert image to array
    image = Image.fromarray(image)    
    draw = ImageDraw.Draw(image)
    draw.rectangle((gt_coords[0], gt_coords[1], gt_coords[0] + CHAR_W, gt_coords[1] + CHAR_H), outline = 'green', width = 2)
    
    if pred_coords:
        
        draw.rectangle((pred_coords[0], pred_coords[1], pred_coords[0] + CHAR_W, pred_coords[1] + CHAR_H), outline = 'red', width = 2)
    
    return image

In [ ]:
#generate an example image
sample_im, pos, _, cat = generate_sample_image()

#get image with bounding box
im = plot_bounding_box(sample_im, pos, (pos[0] + 5, pos[1] - 10))

plt.figure(figsize = (8, 8))
plt.imshow(im)
plt.xticks([])
plt.yticks([])
plt.xlabel('Predicted: ' + cat, color = 'green')
plt.ylabel('True: ' + cat, color = 'green' )
plt.show()

<u> **Creating a function to generate data to the model** </u>

Next we will create a function to feed data to the model. Instead of creating a number of training and testing data we will create a function that feeds data continuously to the model until we reach the desired number of epochs. Every batch is brand new, so the model never sees the same image twice and can't just memorise the training set.

We also **normalise the box coordinates** to be between 0 and 1 by dividing x by the image width and y by the image height. In pixels the box error would be in the thousands (a 100 pixel mistake squared is 10,000!) while the classification loss is below 1, so the box loss would drown out the classification loss. Keeping both targets on a similar scale lets the two heads learn together.

In [ ]:
#data generator function 
def generate_data(batch_size = 32):
    
    while True:
        #create empty arrays for the generated data (float32 is what the model uses anyway)
        x_batch = np.zeros((batch_size, IMG_H, IMG_W, 3), dtype = 'float32')
        y_batch = np.zeros((batch_size, 1), dtype = 'float32')
        boundary_box = np.zeros((batch_size, 2), dtype = 'float32')
        
        for i in range(batch_size):
            
            #generate an example image
            sample_im, pos, person, _ = generate_sample_image()
            
            #put the images to the arrays
            x_batch[i] = sample_im/255 #normalize
            y_batch[i] = person
            boundary_box[i, 0] = pos[0] / IMG_W #normalise x to 0-1
            boundary_box[i, 1] = pos[1] / IMG_H #normalise y to 0-1
            
        yield {'image': x_batch} , {'class': y_batch, 'box': boundary_box}      

<u> **Create the model** </u>

Here we are doing object localization and classification. So our model has to localize the position of the object we are looking for in the picture and then also identify which object is in the picture (Here we have two different people, Waldo and Wilma). 

Both heads share the same convolutional layers (the feature extractor). Four blocks of Conv → MaxPool shrink the 175x250 image down to a small 10x15 grid of features.

For classification ("**what** is in the picture?") the answer is the same wherever the person stands. So we use **global max pooling**: for each of the 32 feature maps we keep only its strongest response, anywhere in the image. Then a small Dense layer with a sigmoid activation gives the probability of Waldo, trained with binary cross entropy.

For localization ("**where** is it?") the position matters, so we can't throw it away with pooling. Instead we **flatten** the 10x15 grid, which keeps the spatial layout, and use a small Dense layer followed by a Dense layer with two units (the normalised x and y of the top-left corner of the box) and no activation. We use the "mean squared error" as the loss.

Keeping the Dense layers small matters a lot: in the first version of this notebook, flattening a 18x27x64 feature map straight into a 1024 unit Dense layer created 32 million weights in that one layer (65 million in total!). This model has about 0.3 million.

In [ ]:
#create the model
def convolutional_block(inputs):
    
    x = tf.keras.layers.Conv2D(8, 3, padding = 'same', activation = 'relu')(inputs)
    x = tf.keras.layers.MaxPool2D(2)(x)
    
    x = tf.keras.layers.Conv2D(16, 3, padding = 'same', activation = 'relu')(x)
    x = tf.keras.layers.MaxPool2D(2)(x)
    
    x = tf.keras.layers.Conv2D(32, 3, padding = 'same', activation = 'relu')(x)
    x = tf.keras.layers.MaxPool2D(2)(x)
        
    x = tf.keras.layers.Conv2D(32, 3, padding = 'same', activation = 'relu')(x)
    x = tf.keras.layers.MaxPool2D(2)(x)
    
    return x

def regression_block(x):
    
    #flatten keeps the position information, which the box head needs
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(64, activation = 'relu')(x)
    x = tf.keras.layers.Dense(2, name = 'box')(x)
    
    return x

def classification_block(x):
    
    #global max pooling: "did this feature fire anywhere in the image?"
    x = tf.keras.layers.GlobalMaxPooling2D()(x)
    x = tf.keras.layers.Dense(32, activation = 'relu')(x)
    x = tf.keras.layers.Dense(1, activation = 'sigmoid', name = 'class')(x)
    
    return x

In [ ]:
#create the model instance
inputs = tf.keras.Input((IMG_H, IMG_W, 3), name = 'image')

#conv block
x = convolutional_block(inputs)

#outputs
box_output = regression_block(x)
class_output = classification_block(x)

#model instance
model = tf.keras.Model(inputs = inputs, outputs = [class_output, box_output])

In [ ]:
#model summary
model.summary()

<u> **Custom callbacks to visualize the predictions & reduce the learning rate** </u>

Next we will create functions to visualize our predicted bounding box & person after every two epochs and schedule the learning rate. 

In [ ]:
#custom function to visualize the predictions after epochs
def test_model():
    
    fig, ax = plt.subplots(1, 3, figsize = (15, 5))    
    
    for i in range(3):
        
        #get sample image
        sample_im, pos, _, cat = generate_sample_image()
        sample_image_normalized = sample_im.reshape(1, IMG_H, IMG_W, 3)/255
        predicted_class, predicted_box = model.predict(sample_image_normalized, verbose = 0)
        
        if predicted_class > 0.5:
            predicted_class = 'Waldo'
        else:
            predicted_class = 'Wilma'
        
        #assign color
        col = 'green' if (predicted_class == cat) else 'red'
        
        #turn the normalised (0-1) box back into pixels
        pred_x = predicted_box[0][0] * IMG_W
        pred_y = predicted_box[0][1] * IMG_H
        
        #get bounding boxes
        im = plot_bounding_box(sample_im, pos, (pred_x, pred_y))
        
        #plot image
        ax[i].imshow(im)
        ax[i].set_xticks([])
        ax[i].set_yticks([])
        ax[i].set_ylabel('True: ' + cat, color = 'green')
        ax[i].set_xlabel('Predicted: ' + predicted_class, color = col)
        
    plt.show()
        
class VisCallback(tf.keras.callbacks.Callback):
    
    def on_epoch_end(self, epoch, logs = None):
        
        if epoch % 2 == 0:
            
            test_model()
            
#learning rate scheduleer
def lr_schedule(epoch, lr):
    
    if (epoch + 1) % 5 == 0:
        
        lr *= 0.2
    
    return max(lr, 3e-7)

<u> **Compile and train the model**</u>

In [ ]:
#compile
#loss_weights: the box loss (normalised coordinates) is small, so we weight it x10
#so the model cares about the box as much as about the class
#learning rate 0.0005 (half the default) makes every run learn reliably
model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate = 5e-4), loss = {'class': 'binary_crossentropy', 'box': 'mse'}, \
              loss_weights = {'class': 1, 'box': 10}, \
              metrics = {'class': 'accuracy', 'box': 'mse'})

In [ ]:
#beginning time
tick = time.time()

#fit the model
hist = model.fit(generate_data(), epochs = 6, steps_per_epoch = 100, callbacks = [VisCallback(),tf.keras.callbacks.LearningRateScheduler(lr_schedule)])

#end time
tock = time.time()

In [ ]:
#print time taken 
print('Took {} minutes to finish training 6 epochs'.format(np.round((tock - tick)/60, 2)))

In [ ]:
#test again in the end a couple of times
for i in range(10):
    test_model()

In the end we can see that the model predictions have reached a high accuracy and the box error has reduced significantly. So when we test the model in the end we see that the model is able to predict the person and the location of the box well.

<u> **Measuring box quality with IoU** </u>

Looking at pictures is a good sanity check, but we can also put a number on it. **Intersection over Union (IoU)** is the area where the predicted and real boxes overlap, divided by the area they cover together. 1 means a perfect match, 0 means no overlap. A common rule of thumb is that IoU > 0.5 counts as a correct detection.

In [ ]:
def iou(box_a, box_b):
    #boxes are top-left corners, all boxes are CHAR_W x CHAR_H
    overlap_w = max(0, CHAR_W - abs(box_a[0] - box_b[0]))
    overlap_h = max(0, CHAR_H - abs(box_a[1] - box_b[1]))
    intersection = overlap_w * overlap_h
    union = 2 * CHAR_W * CHAR_H - intersection
    return intersection / union

#evaluate on 300 new images
x_test, y_test = next(generate_data(batch_size = 300))
pred_class, pred_box = model.predict(x_test, verbose = 0)

accuracy = np.mean((pred_class > 0.5) == y_test['class'])
scale = np.array([IMG_W, IMG_H])
ious = np.array([iou(p * scale, t * scale) for p, t in zip(pred_box, y_test['box'])])

print('Classification accuracy: {:.1%}'.format(accuracy))
print('Mean IoU: {:.2f}'.format(ious.mean()))
print('Detections with IoU > 0.5: {:.1%}'.format(np.mean(ious > 0.5)))

<u> **Testing Unseen images** </u>


In [ ]:
test_im=Image.open('images/16.jpg').convert('RGB')
plot_im = np.array(test_im).astype('uint8')
test_im = test_im.resize((IMG_W, IMG_H))
sample_im=np.array(test_im).astype('uint8')
sample_image_normalized = sample_im.reshape(1, IMG_H, IMG_W, 3)/255
predicted_class, predicted_box = model.predict(sample_image_normalized)

if predicted_class > 0.5:
    predicted_class = 'Waldo'
else:
    predicted_class = 'Wilma'

print(predicted_class)